# Streamlit Interface - Stable Diffusion Studio
## BFGAI_Feri-Putra


## 1. Instalasi Dependencies


In [1]:
!pip install diffusers transformers accelerate pillow matplotlib opencv-python streamlit streamlit-drawable-canvas

  Using cached streamlit-1.60.0-py3-none-any.whl.metadata (10 kB)
  Using cached streamlit_drawable_canvas-0.9.3-py3-none-any.whl.metadata (8.8 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 56.9 MB/s eta 0:00:00


In [3]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("PyTorch version:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

CUDA available: True
PyTorch version: 2.6.0+cu124
GPU: Tesla T4


## 2. Tulis Pipeline Module (`sd_pipeline.py`)


In [3]:
%%writefile sd_pipeline.py
import torch
import numpy as np
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline, StableDiffusionInpaintPipeline, StableDiffusionImg2ImgPipeline
from diffusers import DPMSolverMultistepScheduler, DDIMScheduler, EulerAncestralDiscreteScheduler
from diffusers import LMSDiscreteScheduler, PNDMScheduler
from PIL import Image, ImageDraw
import gc
import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "runwayml/stable-diffusion-v1-5"
INPAINT_MODEL_ID = "runwayml/stable-diffusion-inpainting"

# ============================================================
# CRITERIA 1: TEXT-TO-IMAGE GENERATION
# ============================================================

def load_base_pipe():
    pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
    pipe = pipe.to(DEVICE)
    if DEVICE == "cuda":
        pipe.enable_attention_slicing()
    return pipe

def load_inpaint_pipe():
    pipe = StableDiffusionInpaintPipeline.from_pretrained(INPAINT_MODEL_ID, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
    pipe = pipe.to(DEVICE)
    if DEVICE == "cuda":
        pipe.enable_attention_slicing()
    return pipe

def generate_simple_image(prompt, negative_prompt="", seed=42):
    pipe = load_base_pipe()
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
    ).images[0]
    return image

def generate_advanced_image(prompt, negative_prompt="", seed=42, guidance_scale=7.5, num_inference_steps=50):
    pipe = load_base_pipe()
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]
    return image

def load_scheduler(pipe, scheduler_name):
    scheduler_name = scheduler_name.lower()
    if scheduler_name == "euler a" or scheduler_name == "euler":
        scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "dpm++" or scheduler_name == "dpm++ 2m karras":
        scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "ddim":
        scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "lms":
        scheduler = LMSDiscreteScheduler.from_config(pipe.scheduler.config)
    elif scheduler_name == "pndm":
        scheduler = PNDMScheduler.from_config(pipe.scheduler.config)
    else:
        raise ValueError(f"Scheduler '{scheduler_name}' not recognized. Choose from: Euler A, DPM++, DDIM, LMS, PNDM")
    pipe.scheduler = scheduler
    return pipe

def generate_batch(prompt, negative_prompt="", seed=42, guidance_scale=7.5, num_inference_steps=50, n_images=4, scheduler_name=None):
    pipe = load_base_pipe()
    if scheduler_name:
        pipe = load_scheduler(pipe, scheduler_name)
    images = []
    for i in range(n_images):
        generator = torch.Generator(device=DEVICE).manual_seed(seed + i)
        image = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            generator=generator,
            guidance_scale=guidance_scale,
            num_inference_steps=num_inference_steps,
        ).images[0]
        images.append(image)
    return images

def display_grid(images, rows=2, cols=2, figsize=(10, 10), titles=None):
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i])
            ax.axis('off')
            if titles and i < len(titles):
                ax.set_title(titles[i], fontsize=10)
        else:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

def display_image(image, title=None, figsize=(6, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

# ============================================================
# CRITERIA 2: IMAGE-TO-IMAGE (INPAINTING / OUTPAINTING)
# ============================================================

def inpaint_engine(image, mask, prompt, negative_prompt="", seed=9, guidance_scale=7.5, num_inference_steps=50):
    pipe = load_inpaint_pipe()
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=image,
        mask_image=mask,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
    ).images[0]
    return result

def create_manual_mask(image, mask_region):
    mask = Image.new("L", image.size, 0)
    draw = ImageDraw.Draw(mask)
    if isinstance(mask_region, tuple):
        if len(mask_region) == 4:
            draw.rectangle(mask_region, fill=255)
        elif len(mask_region) == 2:
            cx, cy = mask_region
            r = 40
            draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)
    elif isinstance(mask_region, list):
        for region in mask_region:
            if len(region) == 4:
                draw.rectangle(region, fill=255)
            elif len(region) == 2:
                cx, cy = region
                r = 40
                draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=255)
    return mask

def auto_mask_with_segmentation(image):
    try:
        from transformers import pipeline as hf_pipeline
        segmenter = hf_pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-ade-512-512")
        results = segmenter(image)
        mask = Image.new("L", image.size, 0)
        for r in results:
            if r.get("mask"):
                mask_np = np.array(r["mask"])
                mask_np = (mask_np > 0.5).astype(np.uint8) * 255
                mask_arr = np.maximum(np.array(mask), mask_np)
                mask = Image.fromarray(mask_arr.astype(np.uint8))
        return mask
    except ImportError:
        print("transformers not available for segmentation, falling back to manual masking")
        return None

def prepare_outpainting(image, direction="right", expand_pixels=128, fill_color=(0, 0, 0)):
    w, h = image.size
    if direction == "right":
        new_w = w + expand_pixels
        new_img = Image.new("RGB", (new_w, h), fill_color)
        new_img.paste(image, (0, 0))
        mask = Image.new("L", (new_w, h), 0)
        mask_draw = ImageDraw.Draw(mask)
        mask_draw.rectangle([w, 0, new_w, h], fill=255)
    elif direction == "left":
        new_w = w + expand_pixels
        new_img = Image.new("RGB", (new_w, h), fill_color)
        new_img.paste(image, (expand_pixels, 0))
        mask = Image.new("L", (new_w, h), 0)
        mask_draw = ImageDraw.Draw(mask)
        mask_draw.rectangle([0, 0, expand_pixels, h], fill=255)
    elif direction == "up":
        new_h = h + expand_pixels
        new_img = Image.new("RGB", (w, new_h), fill_color)
        new_img.paste(image, (0, expand_pixels))
        mask = Image.new("L", (w, new_h), 0)
        mask_draw = ImageDraw.Draw(mask)
        mask_draw.rectangle([0, 0, w, expand_pixels], fill=255)
    elif direction == "down":
        new_h = h + expand_pixels
        new_img = Image.new("RGB", (w, new_h), fill_color)
        new_img.paste(image, (0, 0))
        mask = Image.new("L", (w, new_h), 0)
        mask_draw = ImageDraw.Draw(mask)
        mask_draw.rectangle([0, h, w, new_h], fill=255)
    else:
        raise ValueError("direction must be one of: left, right, up, down")
    return new_img, mask

def do_outpainting(image, prompt, direction="right", expand_pixels=128, seed=9, guidance_scale=7.5, num_inference_steps=50):
    expanded_img, mask = prepare_outpainting(image, direction, expand_pixels)
    result = inpaint_engine(expanded_img, mask, prompt, seed=seed, guidance_scale=guidance_scale, num_inference_steps=num_inference_steps)
    return result

def zoom_out(image, prompt, expand_pixels=128, steps=2, seed=9, guidance_scale=7.5, num_inference_steps=50):
    current_img = image
    for i in range(steps):
        directions = ["right", "down", "left", "up"]
        for direction in directions:
            current_img = do_outpainting(
                current_img, prompt, direction=direction,
                expand_pixels=expand_pixels, seed=seed + i,
                guidance_scale=guidance_scale, num_inference_steps=num_inference_steps
            )
    return current_img

def two_stage_generate(prompt, negative_prompt="", seed=42, guidance_scale=7.5, num_inference_steps=50):
    pipe_base = load_base_pipe()
    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    init_image = pipe_base(
        prompt=prompt,
        negative_prompt=negative_prompt,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        denoising_end=0.8,
    ).images[0]

    pipe_refiner = StableDiffusionImg2ImgPipeline.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
    ).to(DEVICE)
    if DEVICE == "cuda":
        pipe_refiner.enable_attention_slicing()

    refined = pipe_refiner(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        generator=generator,
        guidance_scale=guidance_scale,
        num_inference_steps=num_inference_steps,
        strength=0.2,
    ).images[0]

    return refined

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Memory cleared: gc.collect() and torch.cuda.empty_cache() executed.")


Writing sd_pipeline.py


## 3. Tulis Streamlit App (`streamlit_app.py`)


In [8]:
!pip install streamlit==1.40.0 streamlit-drawable-canvas==0.9.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: watchdog
    Found existing installation: watchdog 6.0.0
    Uninstalling watchdog-6.0.0:
      Successfully uninstalled watchdog-6.0.0
  Attempting uninstall: packaging
    Found existing installation: packaging 26.2
    Uninstalling packaging-26.2:
      Successfully uninstalled packaging-26.2
  Attempting uninstall: cachetools
    Found existing installation: cachetools 6.2.6
    Uninstalling cachetools-6.2.6:
      Successfully uninstalled cachetools-6.2.6
  Attempting uninstall: streamlit
    Found existing installation: streamlit 1.60.0
    Uninstalling streamlit-1.60.0:
      Successfully uninstalled streamlit-1.60.0
ERROR: pip's dependency resolve

In [2]:
%%writefile streamlit_app.py
import streamlit as st
import torch
import gc
import numpy as np
from PIL import Image

from sd_pipeline import (
    generate_simple_image, generate_advanced_image,
    generate_batch, display_grid, load_scheduler,
    load_base_pipe, load_inpaint_pipe,
    inpaint_engine, create_manual_mask,
    prepare_outpainting, do_outpainting, zoom_out,
    two_stage_generate, clear_memory, DEVICE
)

st.set_page_config(page_title="Stable Diffusion Studio", layout="wide")

DEFAULT_NEGATIVE = "photorealistic, realistic, photograph, 3d render, messy, blurry, low quality, bad art, ugly, sketch, grainy, unfinished, chromatic aberration"
SCHEDULER_OPTIONS = ["Euler A", "DPM++", "DDIM"]

if "generated_images" not in st.session_state:
    st.session_state.generated_images = []
if "current_image" not in st.session_state:
    st.session_state.current_image = None
if "inpaint_prompt" not in st.session_state:
    st.session_state.inpaint_prompt = "a large severely damaged and broken satellite wreckage, crashed, twisted metal panels, cracked blue solar panels, bent antenna, exposed wiring, scattered metal debris"
if "inpaint_seed" not in st.session_state:
    st.session_state.inpaint_seed = 9
if "inpaint_gs" not in st.session_state:
    st.session_state.inpaint_gs = 10
if "inpaint_steps" not in st.session_state:
    st.session_state.inpaint_steps = 30

st.sidebar.title("Stable Diffusion Studio")
st.sidebar.markdown(f"**Device:** {DEVICE}")
if st.sidebar.button("Clear Memory (GPU)"):
    clear_memory()
    st.sidebar.success("Memory cleared!")

tab1, tab2 = st.tabs(["Text-to-Image", "Inpainting / Outpainting"])

# ═══════════════════════════════════════════════════════════════
# TAB 1
# ═══════════════════════════════════════════════════════════════
with tab1:
    st.header("Text-to-Image Generation")

    col_left, col_right = st.columns([1, 1])

    with col_left:
        prompt = st.text_area("Prompt", value="a beautiful landscape with mountains and a lake at sunset", height=100)
        negative_prompt = st.text_area("Negative Prompt", value=DEFAULT_NEGATIVE, height=80)
        guidance_scale = st.slider("Guidance Scale", 1.0, 20.0, 7.5, 0.5, key="tab1_gs")
        num_inference_steps = st.slider("Inference Steps", 5, 50, 30, 1, key="tab1_steps")
        scheduler_choice = st.selectbox("Scheduler", SCHEDULER_OPTIONS, key="tab1_sched")
        num_images = st.number_input("Number of Images (batch)", 1, 4, 1, 1, key="tab1_nimg")
        seed = st.number_input("Seed", 0, 999999, 222, 1, key="tab1_seed")
        generate_btn = st.button("Generate", type="primary", use_container_width=True, key="tab1_gen")

    with col_right:
        if generate_btn:
            with st.spinner("Generating image(s)..."):
                try:
                    pipe = load_base_pipe()
                    pipe = load_scheduler(pipe, scheduler_choice)

                    if num_images > 1:
                        images = generate_batch(
                            prompt=prompt, negative_prompt=negative_prompt,
                            seed=seed, guidance_scale=guidance_scale,
                            num_inference_steps=num_inference_steps,
                            n_images=num_images, scheduler_name=scheduler_choice
                        )
                        st.session_state.generated_images = images
                        st.session_state.current_image = images[0]

                        grid_cols = st.columns(2)
                        for i, img in enumerate(images):
                            with grid_cols[i % 2]:
                                st.image(img, caption=f"Image {i+1}", use_container_width=True)
                    else:
                        img = generate_advanced_image(
                            prompt=prompt, negative_prompt=negative_prompt,
                            seed=seed, guidance_scale=guidance_scale,
                            num_inference_steps=num_inference_steps
                        )
                        st.session_state.generated_images = [img]
                        st.session_state.current_image = img
                        st.image(img, caption="Generated Image", use_container_width=True)

                    st.success("Generation complete!")
                except Exception as e:
                    st.error(f"Error: {str(e)}")

        if st.session_state.generated_images:
            st.subheader("Last Result")
            st.image(st.session_state.generated_images[0], use_container_width=True)

# ═══════════════════════════════════════════════════════════════
# TAB 2
# ═══════════════════════════════════════════════════════════════
with tab2:
    st.header("Inpainting & Outpainting (Zoom Out)")

    col_left, col_right = st.columns([1, 1])

    with col_left:
        image_source = st.radio("Image Source", ["Last Generated", "Upload"], horizontal=True)

        input_image = None
        if image_source == "Last Generated":
            if st.session_state.current_image is not None:
                input_image = st.session_state.current_image
                st.image(input_image, caption="Input Image", use_container_width=True)
            elif st.session_state.generated_images:
                input_image = st.session_state.generated_images[0]
                st.image(input_image, caption="Input Image (from batch)", use_container_width=True)
            else:
                st.warning("No image available. Upload one instead.")
                uploaded = st.file_uploader("Upload image:", type=["png", "jpg", "jpeg"], key="upload_fallback")
                if uploaded:
                    input_image = Image.open(uploaded).convert("RGB")
                    st.image(input_image, caption="Uploaded Image", use_container_width=True)
        else:
            uploaded = st.file_uploader("Upload image:", type=["png", "jpg", "jpeg"], key="upload_main")
            if uploaded:
                input_image = Image.open(uploaded).convert("RGB")
                st.image(input_image, caption="Uploaded Image", use_container_width=True)

    with col_right:
        st.subheader("Controls")

        canvas_result = None
        try:
            from streamlit_drawable_canvas import st_canvas
            if input_image is not None:
                img_resized = input_image.resize((384, 384))
                canvas_result = st_canvas(
                    fill_color="rgba(255, 0, 0, 0.3)",
                    stroke_width=25,
                    stroke_color="rgba(255, 0, 0, 0.8)",
                    background_image=img_resized,
                    height=384,
                    width=384,
                    drawing_mode="freedraw",
                    key="canvas",
                )
        except ImportError:
            st.warning("streamlit-drawable-canvas not installed. Mask will use center circle.")
            st.info("Install: pip install streamlit-drawable-canvas")
            canvas_result = None

        inpaint_prompt_val = st.text_input("Inpaint Prompt", value=st.session_state.inpaint_prompt, key="tab2_prompt")
        st.session_state.inpaint_prompt = inpaint_prompt_val
        inpaint_negative = st.text_input("Negative Prompt", value=DEFAULT_NEGATIVE, key="tab2_neg")
        inpaint_seed_val = st.number_input("Seed", 0, 999999, st.session_state.inpaint_seed, 1, key="tab2_seed")
        st.session_state.inpaint_seed = inpaint_seed_val
        inpaint_gs_val = st.slider("Guidance Scale", 1.0, 20.0, st.session_state.inpaint_gs, 0.5, key="tab2_gs")
        st.session_state.inpaint_gs = inpaint_gs_val
        inpaint_steps_val = st.slider("Inference Steps", 5, 50, st.session_state.inpaint_steps, 1, key="tab2_steps")
        st.session_state.inpaint_steps = inpaint_steps_val

        inpaint_btn = st.button("Run Inpainting", type="primary", use_container_width=True, key="tab2_inpaint")
        zoom_btn = st.button("Zoom Out (Expand All Sides)", use_container_width=True, key="tab2_zoom")

    if input_image is not None and inpaint_btn:
        with st.spinner("Running inpainting..."):
            try:
                if canvas_result is not None and hasattr(canvas_result, 'image_data') and canvas_result.image_data is not None:
                    mask_arr = canvas_result.image_data.astype(np.uint8)
                    if mask_arr.ndim == 3 and mask_arr.shape[2] >= 4:
                        mask_arr = mask_arr[:, :, 3]
                    elif mask_arr.ndim == 3:
                        mask_arr = mask_arr[:, :, 0]
                    mask_img = Image.fromarray(mask_arr).convert("L")
                    mask_img = mask_img.resize(input_image.size, Image.NEAREST)
                else:
                    cx, cy = input_image.size[0] // 2, input_image.size[1] // 2
                    mask_img = create_manual_mask(input_image, (cx, cy))

                result = inpaint_engine(
                    image=input_image, mask=mask_img,
                    prompt=st.session_state.inpaint_prompt,
                    negative_prompt=inpaint_negative,
                    seed=st.session_state.inpaint_seed,
                    guidance_scale=st.session_state.inpaint_gs,
                    num_inference_steps=st.session_state.inpaint_steps
                )
                st.session_state.current_image = result

                res_col1, res_col2 = st.columns(2)
                with res_col1:
                    st.image(input_image, caption="Original", use_container_width=True)
                    st.image(mask_img, caption="Mask", use_container_width=True)
                with res_col2:
                    st.image(result, caption="Inpainted", use_container_width=True)
                st.success("Inpainting complete!")
            except Exception as e:
                st.error(f"Inpainting error: {str(e)}")

    if input_image is not None and zoom_btn:
        with st.spinner("Running zoom out (outpainting all directions)..."):
            try:
                zoom_result = zoom_out(
                    image=input_image,
                    prompt=st.session_state.inpaint_prompt,
                    expand_pixels=64, steps=1,
                    seed=st.session_state.inpaint_seed,
                    guidance_scale=st.session_state.inpaint_gs,
                    num_inference_steps=st.session_state.inpaint_steps
                )
                st.session_state.current_image = zoom_result

                res_col1, res_col2 = st.columns(2)
                with res_col1:
                    st.image(input_image, caption="Original", use_container_width=True)
                with res_col2:
                    st.image(zoom_result, caption="Zoom Out Result", use_container_width=True)
                st.success("Zoom Out complete!")
            except Exception as e:
                st.error(f"Zoom Out error: {str(e)}")

st.markdown("---")
st.markdown("Stable Diffusion Studio | Pipeline: runwayml/stable-diffusion-v1-5 & runwayml/stable-diffusion-inpainting")


Overwriting streamlit_app.py


## 4. Verifikasi File


In [3]:
import os
files = ["sd_pipeline.py", "streamlit_app.py"]
for f in files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    print(f"{'OK' if exists else 'MISSING'} {f} ({size} bytes)")


OK sd_pipeline.py (9893 bytes)
OK streamlit_app.py (11369 bytes)


## 5. Jalankan Streamlit dengan Ngrok

Uncomment dan jalankan cell di bawah untuk mengekspos aplikasi melalui ngrok.


In [ ]:
!pip install pyngrok
from pyngrok import ngrok
import subprocess, time, sys, os

ngrok.set_auth_token("3GzVJA5diEnH0sVVH4mmMjp5MgA_6aiJ8pWUJtcqWXcapsG84")

# Kill existing streamlit processes
import os
os.system("pkill -f streamlit 2>/dev/null || true")
time.sleep(1)

# Run Streamlit in background
subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "streamlit_app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--browser.gatherUsageStats", "false"
])
time.sleep(8)

# Create ngrok tunnel
public_url = ngrok.connect(8501)
print(f"Streamlit app running at: {public_url}")
print("Record a 1-5 minute demo video showing the interface.")


Streamlit app running at: NgrokTunnel: "https://outline-gaffe-octopus.ngrok-free.dev" -> "http://localhost:8501"
Record a 1-5 minute demo video showing the interface.


In [7]:
print("Notebook Streamlit siap!")
print("1. Jalankan cell instalasi dependencies")
print("2. Jalankan cell untuk menulis sd_pipeline.py")
print("3. Jalankan cell untuk menulis streamlit_app.py")
print("4. Uncomment dan jalankan cell ngrok")
print("5. Rekam demo video (1-5 menit, .mp4)")


Notebook Streamlit siap!
1. Jalankan cell instalasi dependencies
2. Jalankan cell untuk menulis sd_pipeline.py
3. Jalankan cell untuk menulis streamlit_app.py
4. Uncomment dan jalankan cell ngrok
5. Rekam demo video (1-5 menit, .mp4)


---
## Demo Video
Rekam aplikasi selama 1-5 menit dan simpan sebagai `video_demo_aplikasi_BFGAI.mp4`.

Tampilkan:
1. Text-to-Image tab dengan prompt, negative prompt, slider, generate
2. Scheduler selection (Euler A, DPM++, DDIM)
3. Batch generation (4 gambar)
4. Inpainting dengan drawable canvas mask
5. Zoom Out
